# Day 1-04｜Roboflow BBOX Dataset 與 YOLO Detection 訓練準備
> Python 籃球運動資料分析課程  
> 本單元定義籃球偵測類別、Roboflow YOLO export 放置位置，並用已訓練 YOLO detector 檢查實際影片 frame。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 建立 detection dataset 的類別規格與資料夾結構。
- 檢查 Roboflow YOLO detection export 是否可被 Ultralytics 讀取。
- 使用教師提供的 YOLO detector 權重產生實際偵測結果。

## 資料放置位置
- Roboflow YOLO detection export：`assets/datasets/roboflow_bbox_yolo/`
- 已訓練 detector 權重：`assets/models/detectors/yolo26n_basketball_player_best.pt`


## 課程流程
1. 確認籃球偵測類別與 Roboflow export 結構。
2. 檢查 `assets/datasets/roboflow_bbox_yolo/`。
3. 閱讀實際訓練程式；預設不重新訓練。
4. 使用已訓練 detector 對參考影片 frame 執行推論。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


## Roboflow YOLO Detection Export 格式
Roboflow 匯出 YOLO 格式後，請將整個 dataset 放在：

```text
assets/datasets/roboflow_bbox_yolo/
├── data.yaml
├── train/ 或 images/train + labels/train
├── valid/ 或 images/valid + labels/valid
└── test/  或 images/test  + labels/test
```

本課程使用的 detector 類別與參考專案一致：`ball`、`ball-in-basket`、`number`、`player`、`player-in-possession`、`player-jump-shot`、`player-layup-dunk`、`player-shot-block`、`referee`、`rim`。


In [ ]:
from src.roboflow_utils import DETECTION_CLASSES, dataset_status

BBOX_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_bbox_yolo"
DETECTOR_MODEL_PATH = (
    COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
)

for i, name in enumerate(DETECTION_CLASSES):
    print(i, name)

print("\ndataset status:")
print(dataset_status(BBOX_DATASET_DIR))
print("\nmodel exists:", DETECTOR_MODEL_PATH.exists(), DETECTOR_MODEL_PATH)


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO

    data_yaml = BBOX_DATASET_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 data.yaml。請先將 Roboflow YOLO detection export 放到 assets/datasets/roboflow_bbox_yolo/"
        )

    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=960,
        batch=2,
        workers=4,
        patience=30,
        close_mosaic=10,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "bbox_detection"),
        name="yolo26n_basketball_player",
        plots=True,
    )
    print(results)
else:
    print("RUN_TRAINING=False；本課程預設使用 assets/models/detectors/ 內的已訓練權重。")


In [ ]:
import pandas as pd

from src.cv_utils import save_image_rgb, save_json, show_image
from src.yolo_utils import (
    detector_model_path,
    rgb_from_ultralytics_plot,
    first_reference_video,
    read_video_frame,
    records_to_dicts,
    run_detector_on_image,
)

video_path = first_reference_video(COURSE_ROOT)
frame = read_video_frame(video_path, frame_index=15)
detections, result = run_detector_on_image(
    detector_model_path(COURSE_ROOT),
    frame,
    conf=0.25,
    imgsz=960,
    frame_index=15,
)
vis = rgb_from_ultralytics_plot(result)
show_image(vis, "YOLO detections on reference frame")

out_image = COURSE_ROOT / "assets" / "results" / "d1_04_yolo_detections.png"
out_json = COURSE_ROOT / "assets" / "results" / "d1_04_yolo_detections.json"
save_image_rgb(out_image, vis)
save_json(records_to_dicts(detections), out_json)

print("video:", video_path)
print("detections:", len(detections))
print("saved:", out_image)
pd.DataFrame(records_to_dicts(detections)).head()
